# Geometric EEG SSL — Colab Pretraining Notebook

**What this does:**
1. Installs dependencies
2. Mounts Google Drive (checkpoints saved there — survive session disconnects)
3. Arms a keep-alive against laptop sleep + idle disconnect
4. Clones the repo
5. Downloads PhysioNet MI data via MNE — **resume-aware**, safe to re-run after disconnect
6. Pretrains G1 (geometric, 105 subjects, 100 epochs)
7. Runs the linear probe (LOSO, 105 subjects)
8. Optionally: runs G2, G3, transductive codex ablations

**Before running:** Runtime → Change runtime type → GPU (T4 free, A100 Colab Pro)

---

## 0. GPU check

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('WARNING: no GPU detected — set Runtime → Change runtime type → GPU')
    sys.exit(1)

## 1. Install dependencies

In [ ]:
%%capture
!pip install mne moabb scikit-learn pyyaml

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/geometric_eeg_ssl'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# MNE data cached here — avoids re-downloading across sessions
MNE_DATA_DIR = f'{DRIVE_ROOT}/mne_data'
os.makedirs(MNE_DATA_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Checkpoints saved here
CKPT_ROOT = f'{DRIVE_ROOT}/runs'
os.makedirs(CKPT_ROOT, exist_ok=True)
print(f'Drive mounted. Checkpoints → {CKPT_ROOT}')

## 2b. Keep Colab alive while your laptop sleeps

Colab sessions are tied to the browser tab that owns them. If the laptop sleeps
or the network drops, the WebSocket dies and the runtime can be reclaimed even
though GPU compute is still busy. The training loop saves a checkpoint every 10
epochs, so a disconnect costs at most ~10 epochs — but it's better to avoid one
entirely. Three layers of protection, applied together:

**1. Stop the laptop from sleeping.**
- **macOS:** open Terminal and run `caffeinate -dis &` before closing the lid.
  Or System Settings → Battery → "Prevent automatic sleeping on power adapter
  when the display is off" + plug in. (The `Amphetamine` app is the GUI version.)
- **Windows:** Settings → System → Power → Screen and sleep → set "When plugged
  in, put my device to sleep" to *Never*. Close the lid action: *Do nothing*.

**2. Suppress Colab's idle prompt (cell below).**
Colab pops up a "Are you still here?" dialog after ~90 min of UI inactivity.
The JS snippet below auto-clicks the connect button every minute. Run it in the
notebook (it injects into the Colab page) **before** starting training.

**3. Trust the resume cell.**
Even with the above, a long run might disconnect (Colab free tier has a hard
12-hour cap; Pro is ~24 h). The `RESUME CELL` in section 5 handles this:
it finds the latest `epoch_NNNN.pt` on Drive and restarts training from there.

**Colab Pro background execution** (paid) lets the notebook keep running with
the tab closed — the cleanest solution if you have access.

In [ ]:
# Keep Colab from disconnecting on idle. Paste-and-run; safe to re-execute.
# Auto-clicks the connect button every 60s. Only useful while the tab is open
# in a browser that hasn't been put to sleep by the OS.
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect() {
  const btn = document.querySelector("colab-connect-button");
  if (btn && btn.shadowRoot) {
    const inner = btn.shadowRoot.querySelector("#connect");
    if (inner) inner.click();
  }
  console.log("colab keep-alive ping " + new Date().toLocaleTimeString());
}
if (window._colabKeepAlive) clearInterval(window._colabKeepAlive);
window._colabKeepAlive = setInterval(ClickConnect, 60000);
console.log("colab keep-alive armed (60s interval)");
'''))
print('Keep-alive armed. Re-run this cell after any browser refresh.')


## 3. Clone repo

In [ ]:
import os, sys
REPO_DIR = '/content/geometric-eeg-ssl'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/tianxin-scu/geometric-eeg-ssl.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
print('Repo ready at', REPO_DIR)

## 4. Download PhysioNet MI data

First run takes ~20 min. Subsequent runs load from Drive instantly.

In [ ]:
import os, mne
mne.set_log_level('WARNING')

# PhysioNet MI: 109 subjects, 4 excluded (88, 92, 100, 104)
EXCLUDED = {88, 92, 100, 104}
ALL_SUBJECTS = [s for s in range(1, 110) if s not in EXCLUDED]  # 105 subjects
MI_RUNS = [4, 6, 8, 10, 12, 14]

# MNE caches PhysioNet MI at:
#   {MNE_DATA}/MNE-eegbci-data/files/eegmmidb/1.0.0/S{NNN}/S{NNN}R{RR}.edf
EEGBCI_ROOT = os.path.join(MNE_DATA_DIR, 'MNE-eegbci-data', 'files', 'eegmmidb', '1.0.0')

def subject_fully_cached(subj: int, runs: list[int]) -> bool:
    subj_dir = os.path.join(EEGBCI_ROOT, f'S{subj:03d}')
    if not os.path.isdir(subj_dir):
        return False
    for run in runs:
        edf = os.path.join(subj_dir, f'S{subj:03d}R{run:02d}.edf')
        if not os.path.exists(edf) or os.path.getsize(edf) == 0:
            return False
    return True

# Resume-aware download: skip subjects whose EDF files are already on Drive.
# Safe to re-run after a session reconnect — fully-cached subjects skip in O(stat).
cached = [s for s in ALL_SUBJECTS if subject_fully_cached(s, MI_RUNS)]
todo = [s for s in ALL_SUBJECTS if s not in cached]
print(f'Cached: {len(cached)}/{len(ALL_SUBJECTS)} subjects. To download: {len(todo)}.')

for i, subj in enumerate(todo):
    try:
        mne.datasets.eegbci.load_data(subj, MI_RUNS, path=MNE_DATA_DIR, verbose=False)
    except Exception as e:
        print(f'  subject {subj:3d}: download failed ({e}); will retry on next run')
        continue
    if (i + 1) % 5 == 0 or (i + 1) == len(todo):
        print(f'  downloaded {i+1}/{len(todo)} (subject {subj:3d})')

# Final verification
still_missing = [s for s in ALL_SUBJECTS if not subject_fully_cached(s, MI_RUNS)]
if still_missing:
    print(f'WARNING: {len(still_missing)} subjects still incomplete: {still_missing}')
    print('Re-run this cell to resume; partial files are skipped automatically.')
else:
    print(f'All {len(ALL_SUBJECTS)} subjects ready under {EEGBCI_ROOT}')


## 5. Pretrain — G1 (geometric, score-only bias)

~7–10 hrs on T4, ~2–3 hrs on A100. Checkpoints saved every 10 epochs to Drive.
If the session disconnects, resume with `--resume epoch_NNNN.pt`.

In [16]:
G1_CKPT_DIR = f'{CKPT_ROOT}/g1_full'
os.makedirs(G1_CKPT_DIR, exist_ok=True)

# Point MNE_DATA at Drive so the loader finds the cached EDF files
os.environ['MNE_DATA'] = MNE_DATA_DIR

!python {REPO_DIR}/scripts/pretrain.py \
    --config {REPO_DIR}/configs/pretrain/geometric_g1.yaml \
    --ckpt-dir {G1_CKPT_DIR} \
    --device cuda \
    2>&1 | tee {G1_CKPT_DIR}/train_log.txt

device: cuda
loading 105 subjects …
  subject   1 run  4: 15 epochs
  subject   1 run  8: 15 epochs
  subject   1 run 12: 15 epochs
  subject   1 run  6: 15 epochs
  subject   1 run 10: 15 epochs
  subject   1 run 14: 15 epochs
  subject   2 run  4: 15 epochs
  subject   2 run  8: 15 epochs
  subject   2 run 12: 15 epochs
  subject   2 run  6: 15 epochs
  subject   2 run 10: 15 epochs
  subject   2 run 14: 15 epochs
  subject   3 run  4: 15 epochs
  subject   3 run  8: 15 epochs
  subject   3 run 12: 15 epochs
  subject   3 run  6: 15 epochs
  subject   3 run 10: 15 epochs
  subject   3 run 14: 15 epochs
  subject   4 run  4: 15 epochs
  subject   4 run  8: 15 epochs
  subject   4 run 12: 15 epochs
  subject   4 run  6: 15 epochs
  subject   4 run 10: 15 epochs
  subject   4 run 14: 15 epochs
  subject   5 run  4: 15 epochs
  subject   5 run  8: 15 epochs
  subject   5 run 12: 15 epochs
  subject   5 run  6: 15 epochs
  subject   5 run 10: 15 epochs
  subject   5 run 14: 15 epochs
  su

In [17]:
# ── RESUME CELL (run only if session disconnected) ─────────────────────────
# Find latest checkpoint, then resume.
import glob
ckpts = sorted(glob.glob(f'{G1_CKPT_DIR}/epoch_*.pt'))
if ckpts:
    latest = os.path.basename(ckpts[-1])
    print(f'Resuming from {latest}')
    os.environ['MNE_DATA'] = MNE_DATA_DIR
    !python {REPO_DIR}/scripts/pretrain.py \
        --config {REPO_DIR}/configs/pretrain/geometric_g1.yaml \
        --ckpt-dir {G1_CKPT_DIR} \
        --device cuda \
        --resume {latest} \
        2>&1 | tee -a {G1_CKPT_DIR}/train_log.txt
else:
    print('No checkpoints found — run the pretrain cell above first.')

Resuming from epoch_0099.pt
device: cuda
loading 105 subjects …
  subject   1 run  4: 15 epochs
  subject   1 run  8: 15 epochs
  subject   1 run 12: 15 epochs
  subject   1 run  6: 15 epochs
  subject   1 run 10: 15 epochs
  subject   1 run 14: 15 epochs
  subject   2 run  4: 15 epochs
  subject   2 run  8: 15 epochs
  subject   2 run 12: 15 epochs
  subject   2 run  6: 15 epochs
  subject   2 run 10: 15 epochs
  subject   2 run 14: 15 epochs
  subject   3 run  4: 15 epochs
  subject   3 run  8: 15 epochs
  subject   3 run 12: 15 epochs
  subject   3 run  6: 15 epochs
  subject   3 run 10: 15 epochs
  subject   3 run 14: 15 epochs
  subject   4 run  4: 15 epochs
  subject   4 run  8: 15 epochs
  subject   4 run 12: 15 epochs
  subject   4 run  6: 15 epochs
  subject   4 run 10: 15 epochs
  subject   4 run 14: 15 epochs
  subject   5 run  4: 15 epochs
  subject   5 run  8: 15 epochs
  subject   5 run 12: 15 epochs
  subject   5 run  6: 15 epochs
  subject   5 run 10: 15 epochs
  subjec

## 6. Linear probe — G1

In [18]:
import glob, os
ckpts = sorted(glob.glob(f'{G1_CKPT_DIR}/epoch_*.pt'))
assert ckpts, 'No G1 checkpoint found — run Section 5 first.'
G1_FINAL_CKPT = ckpts[-1]
print(f'Probing checkpoint: {G1_FINAL_CKPT}')

os.environ['MNE_DATA'] = MNE_DATA_DIR
!python {REPO_DIR}/scripts/probe.py \
    --checkpoint {G1_FINAL_CKPT} \
    --output {G1_CKPT_DIR}/probe_results.json \
    --device cuda \
    2>&1 | tee {G1_CKPT_DIR}/probe_log.txt

Probing checkpoint: /content/drive/MyDrive/geometric_eeg_ssl/runs/g1_full/epoch_0099.pt
device: cuda
Discovering n_electrodes from physionet_mi subject 1 ...
  n_electrodes = 64
backbone parameters: 4,770,128  (frozen)
LOSO over 105 subjects on physionet_mi ...
  subject 1: 90 epochs
  subject 2: 90 epochs
  subject 3: 90 epochs
  subject 4: 90 epochs
  subject 5: 90 epochs
  subject 6: 90 epochs
  subject 7: 90 epochs
  subject 8: 90 epochs
  subject 9: 90 epochs
  subject 10: 90 epochs
  subject 11: 90 epochs
  subject 12: 90 epochs
  subject 13: 90 epochs
  subject 14: 90 epochs
  subject 15: 90 epochs
  subject 16: 90 epochs
  subject 17: 90 epochs
  subject 18: 90 epochs
  subject 19: 90 epochs
  subject 20: 90 epochs
  subject 21: 90 epochs
  subject 22: 90 epochs
  subject 23: 90 epochs
  subject 24: 90 epochs
  subject 25: 90 epochs
  subject 26: 90 epochs
  subject 27: 90 epochs
  subject 28: 90 epochs
  subject 29: 90 epochs
  subject 30: 90 epochs
  subject 31: 90 epochs
  s

In [19]:
import json
with open(f'{G1_CKPT_DIR}/probe_results.json') as f:
    res = json.load(f)
agg = res['aggregate']
print(f"G1  BAC = {agg['bac_mean']:.3f} ± {agg['bac_std']:.3f}  (n={agg['n_subjects']} subjects)")

G1  BAC = 0.342 ± 0.074  (n=105 subjects)


## 7. Ablation runs (G2, G3, Transductive Codex)

Run these after G1 completes. Each writes to its own Drive subdirectory.
Skip any you don't need — G1 is the headline result.

In [ ]:
ABLATIONS = [
    ('g2_full', 'geometric_g2.yaml'),
    ('g3_full', 'geometric_g3.yaml'),
    ('codex_full', 'transductive_codex.yaml'),
]

for run_name, config_file in ABLATIONS:
    ckpt_dir = f'{CKPT_ROOT}/{run_name}'
    os.makedirs(ckpt_dir, exist_ok=True)
    config_path = f'{REPO_DIR}/configs/pretrain/{config_file}'
    print(f'\n=== {run_name} ===')
    os.environ['MNE_DATA'] = MNE_DATA_DIR
    !python {REPO_DIR}/scripts/pretrain.py \
        --config {config_path} \
        --ckpt-dir {ckpt_dir} \
        --device cuda \
        2>&1 | tee {ckpt_dir}/train_log.txt

    import glob
    ckpts = sorted(glob.glob(f'{ckpt_dir}/epoch_*.pt'))
    if ckpts:
        os.environ['MNE_DATA'] = MNE_DATA_DIR
        !python {REPO_DIR}/scripts/probe.py \
            --checkpoint {ckpts[-1]} \
            --output {ckpt_dir}/probe_results.json \
            --device cuda \
            2>&1 | tee {ckpt_dir}/probe_log.txt


=== g2_full ===
device: cuda
loading 105 subjects …
  subject   1 run  4: 15 epochs
  subject   1 run  8: 15 epochs
  subject   1 run 12: 15 epochs
  subject   1 run  6: 15 epochs
  subject   1 run 10: 15 epochs
  subject   1 run 14: 15 epochs
  subject   2 run  4: 15 epochs
  subject   2 run  8: 15 epochs
  subject   2 run 12: 15 epochs
  subject   2 run  6: 15 epochs
  subject   2 run 10: 15 epochs
  subject   2 run 14: 15 epochs
  subject   3 run  4: 15 epochs
  subject   3 run  8: 15 epochs
  subject   3 run 12: 15 epochs
  subject   3 run  6: 15 epochs
  subject   3 run 10: 15 epochs
  subject   3 run 14: 15 epochs
  subject   4 run  4: 15 epochs
  subject   4 run  8: 15 epochs
  subject   4 run 12: 15 epochs
  subject   4 run  6: 15 epochs
  subject   4 run 10: 15 epochs
  subject   4 run 14: 15 epochs
  subject   5 run  4: 15 epochs
  subject   5 run  8: 15 epochs
  subject   5 run 12: 15 epochs
  subject   5 run  6: 15 epochs
  subject   5 run 10: 15 epochs
  subject   5 run 1

## 8. Results summary

In [ ]:
import json, glob

all_runs = [
    ('G1 (geometric, score-bias)', f'{CKPT_ROOT}/g1_full'),
    ('G2 (geometric, value-mod)',  f'{CKPT_ROOT}/g2_full'),
    ('G3 (geometric, both)',       f'{CKPT_ROOT}/g3_full'),
    ('Transductive codex',         f'{CKPT_ROOT}/codex_full'),
]

print(f"{'Variant':<30} {'BAC mean':>10} {'BAC std':>10} {'n_subj':>8}")
print('-' * 62)
for label, ckpt_dir in all_runs:
    result_file = f'{ckpt_dir}/probe_results.json'
    if not os.path.exists(result_file):
        print(f'{label:<30} {"(not run yet)":>10}')
        continue
    with open(result_file) as f:
        r = json.load(f)['aggregate']
    print(f"{label:<30} {r['bac_mean']:>10.3f} {r['bac_std']:>10.3f} {r['n_subjects']:>8}")